# 03 — Model 2: DenseNet-121 Transfer Learning (CheXNet)

Fine-tune DenseNet-121 (pretrained on ImageNet) for chest X-ray classification.

**Architecture decisions documented here:**
- DenseNet-121 backbone: dense connections maximize feature reuse (Huang et al., 2017)
- ImageNet pretraining: low-level features (edges, textures) transfer well to medical imaging
- Two-phase training:
  - Phase 1: Freeze backbone, train classifier heads only (prevents catastrophic forgetting)
  - Phase 2: Unfreeze all layers with reduced LR (end-to-end fine-tuning)
- CheXNet reference: Rajpurkar et al. (2017) achieved radiologist-level performance with this setup

In [ ]:
import sys
sys.path.insert(0, "..")

import torch
import yaml
from pathlib import Path
from src.training.trainer import train, set_seed

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Load config
with open("../configs/transfer.yaml") as f:
    config = yaml.safe_load(f)

config["data"]["data_dir"] = "../data"
print(yaml.dump(config, default_flow_style=False))

In [ ]:
# Model info
from src.models.densenet_transfer import CheXVisionDenseNet

model = CheXVisionDenseNet(pretrained=True, freeze_backbone=True)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"Total parameters: {total_params:,}")
print(f"Trainable (Phase 1): {trainable_params:,}")
print(f"Frozen (Phase 1): {frozen_params:,}")
print(f"\nAfter unfreezing, all {total_params:,} parameters become trainable.")

In [ ]:
# Train
import logging
logging.basicConfig(level=logging.INFO)

train(config)

In [ ]:
# Compare with scratch model
# TODO: Load both checkpoints and run comparison
# from src.training.evaluate import compare_models